# SQA QUBO Solver — FPGA Demo

This notebook demonstrates the `SQASolver` high-level interface on the PYNQ-Z2 board.
It covers three ready-to-run problem types:

1. **Number Partition** — split a set of numbers into two equal-sum subsets
2. **Max-Cut** — find the graph partition that maximises edge crossings
3. **Custom QUBO** — plug in your own coupling matrix

**Before running**, make sure these files are in `/home/xilinx/Quantum Annealing Simulation/`:
```
SQA_Opt5.bit
SQA_Opt5.hwh
sqa_solver.py
```

## Setup

In [ ]:
import sys
sys.path.append('/home/xilinx/Quantum Annealing Simulation')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sqa_solver import SQASolver, SQASimulator, qubo_energy

In [ ]:
# Load the FPGA overlay — flashes the bitstream (~5 s)
BITFILE = '/home/xilinx/Quantum Annealing Simulation/SQA_Opt5.bit'
solver = SQASolver(BITFILE, num_trotters=8)
print('Overlay loaded.')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def run_and_report(solver, Q, label, iters=500, beta_start=1/4096,
                   beta_end=8.0, gamma_start=8.0, seed=0):
    """Run the solver, print a summary, and return the SQAResult."""
    n = Q.shape[0]
    print(f'\n{'─'*60}')
    print(f'  {label}  |  {n} spins  |  {iters} iterations')
    print(f'{'─'*60}')

    result = solver.solve(Q, iters=iters, beta_start=beta_start,
                          beta_end=beta_end, gamma_start=gamma_start,
                          seed=seed)

    print(f'  Best energy : {result.best_energy:.6f}')
    print(f'  Best sample : {result.best_sample.astype(int)}')
    print(f'  Runtime     : {result.timing_s:.2f} s')
    print(f'  All trotter energies: {np.round(result.all_energies, 4)}')
    return result


def plot_trotter_energies(result, Q, label):
    """Bar chart of final-state QUBO energies across trotters."""
    nt = result.all_samples.shape[0]
    energies = result.all_energies
    best_t = int(np.argmin(energies))

    colors = ['steelblue'] * nt
    colors[best_t] = 'tomato'

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(nt), energies, color=colors)
    ax.set_xlabel('Trotter index')
    ax.set_ylabel('QUBO energy')
    ax.set_title(f'{label} — final trotter energies (red = best)')
    ax.set_xticks(range(nt))
    plt.tight_layout()
    plt.show()

---
## Problem 1 — Number Partition

**Goal:** given a list of numbers `w`, find a binary assignment `x ∈ {0,1}ⁿ`
that minimises `(sum_A − sum_B)²`, where set A = elements with `x=0`,
set B = elements with `x=1`.

**QUBO formulation:**
```
Q[i,j] = 4·wᵢ·wⱼ            (i ≠ j)
Q[i,i] = 4·wᵢ·(wᵢ − W)      (W = Σ wᵢ)
```
The constant `W²` is dropped (does not affect which `x` minimises the energy).

In [ ]:
# ── Problem definition ────────────────────────────────────────────────────────
# Change these numbers to your own instance (up to 1024 values).
w = np.array([3, 7, 12, 5, 9, 8, 2, 6, 11, 4, 1, 10], dtype=float)

W = w.sum()
n = len(w)
Q_partition = np.outer(4 * w, w)
Q_partition[np.arange(n), np.arange(n)] = 4 * w * (w - W)

print(f'Numbers : {w}')
print(f'Total   : {W}')
print(f'Ideal half-sum: {W/2}')

In [ ]:
# ── Run on FPGA ───────────────────────────────────────────────────────────────
r_part = run_and_report(solver, Q_partition, 'Number Partition', iters=500)

In [ ]:
# ── Decode and display the partition ─────────────────────────────────────────
x = r_part.best_sample.astype(int)
set_A = w[x == 0]
set_B = w[x == 1]

print(f'Set A : {set_A}   sum = {set_A.sum()}')
print(f'Set B : {set_B}   sum = {set_B.sum()}')
print(f'Difference : {abs(set_A.sum() - set_B.sum())}')

plot_trotter_energies(r_part, Q_partition, 'Number Partition')

In [ ]:
# ── Sanity check: brute-force optimum for small instances (n ≤ 20) ───────────
if n <= 20:
    best_bf = np.inf
    best_x_bf = None
    for bits in range(2**n):
        x_bf = np.array([(bits >> k) & 1 for k in range(n)], dtype=float)
        e = qubo_energy(Q_partition, x_bf)
        if e < best_bf:
            best_bf, best_x_bf = e, x_bf
    gap = r_part.best_energy - best_bf
    print(f'Brute-force optimum : {best_bf:.4f}')
    print(f'FPGA result         : {r_part.best_energy:.4f}')
    print(f'Optimality gap      : {gap:.4f}  ({"OPTIMAL" if gap < 1e-3 else "suboptimal"})')
else:
    print('Instance too large for brute force — skipping.')

---
## Problem 2 — Max-Cut

**Goal:** given an undirected weighted graph with adjacency matrix `A`,
find a partition `x ∈ {0,1}ⁿ` that **maximises** the total weight of
edges crossing the cut (one endpoint in each subset).

**QUBO formulation** (minimise the negative cut):
```
Q[i,j] = A[i,j]          (i ≠ j)   ← penalises same-side neighbours
Q[i,i] = −Σⱼ A[i,j]               ← rewards putting node i in any side
```
Minimising `xᵀQx` is equivalent to maximising the cut weight.

In [ ]:
# ── Problem definition ────────────────────────────────────────────────────────
# Adjacency matrix — change to your graph (n ≤ 1024 nodes).
# Example: a cycle graph on 8 nodes with one extra diagonal edge.
n_mc = 8
A = np.zeros((n_mc, n_mc))
for i in range(n_mc):
    A[i, (i+1) % n_mc] = 1
    A[(i+1) % n_mc, i] = 1
A[0, 4] = 1; A[4, 0] = 1   # one diagonal chord

degrees = A.sum(axis=1)
Q_maxcut = A - np.diag(degrees)

print('Adjacency matrix:')
print(A.astype(int))
print(f'\nEdges: {int(A.sum())//2}   Nodes: {n_mc}')

In [ ]:
# ── Run on FPGA ───────────────────────────────────────────────────────────────
r_mc = run_and_report(solver, Q_maxcut, 'Max-Cut', iters=500)

In [ ]:
# ── Decode and display the cut ────────────────────────────────────────────────
x_mc = r_mc.best_sample.astype(int)
cut_weight = sum(
    A[i, j]
    for i in range(n_mc)
    for j in range(i+1, n_mc)
    if x_mc[i] != x_mc[j]
)
print(f'Partition S0 (x=0): nodes {np.where(x_mc == 0)[0].tolist()}')
print(f'Partition S1 (x=1): nodes {np.where(x_mc == 1)[0].tolist()}')
print(f'Cut weight: {cut_weight:.0f}')

plot_trotter_energies(r_mc, Q_maxcut, 'Max-Cut')

In [ ]:
# ── Sanity check: brute-force optimum for small graphs (n ≤ 20) ──────────────
if n_mc <= 20:
    best_cut = 0
    for bits in range(2**n_mc):
        x_bf = np.array([(bits >> k) & 1 for k in range(n_mc)])
        cut = sum(A[i,j] for i in range(n_mc) for j in range(i+1,n_mc)
                  if x_bf[i] != x_bf[j])
        best_cut = max(best_cut, cut)
    print(f'Brute-force max cut : {best_cut:.0f}')
    print(f'FPGA cut            : {cut_weight:.0f}')
    print(f'Optimality          : {"OPTIMAL" if cut_weight >= best_cut - 1e-6 else "suboptimal"}')
else:
    print('Graph too large for brute force — skipping.')

---
## Problem 3 — Custom QUBO

Replace `Q_custom` below with your own `(n × n)` matrix and run the cell.

The matrix can be:
- **symmetric** — `Q[i,j]` is the coupling between variables `i` and `j`; `Q[i,i]` is the local bias
- **upper triangular** — the solver symmetrises it internally

Constraints:
- `n ≤ 1024`  (MAX_NSPIN of the Opt5 kernel)
- entries should be `float32`-range values (no `inf` or `nan`)

In [ ]:
# ── Define your QUBO here ─────────────────────────────────────────────────────
#
# Q[i,j]  = coupling between variable i and j  (negative = want them equal)
# Q[i,i]  = local bias on variable i           (negative = prefer x_i = 1)
#
# Example: a small weighted MAX-2-SAT-like instance on 6 variables.

n_custom = 6
rng = np.random.default_rng(seed=0)
Q_custom = rng.uniform(-1, 1, size=(n_custom, n_custom))
Q_custom = (Q_custom + Q_custom.T) / 2   # make it symmetric
np.fill_diagonal(Q_custom, rng.uniform(-2, 0, size=n_custom))  # negative bias

print('Custom Q matrix:')
print(np.round(Q_custom, 3))

In [ ]:
# ── Annealing parameters ──────────────────────────────────────────────────────
# Tune these for your problem. Start with the defaults below;
# if the solution is poor try increasing iters or beta_end.

ITERS       = 500
BETA_START  = 1 / 4096
BETA_END    = 8.0
GAMMA_START = 8.0

In [ ]:
# ── Run on FPGA ───────────────────────────────────────────────────────────────
r_custom = run_and_report(
    solver, Q_custom, 'Custom QUBO',
    iters=ITERS, beta_start=BETA_START, beta_end=BETA_END, gamma_start=GAMMA_START
)

In [ ]:
# ── Display results ───────────────────────────────────────────────────────────
print('Best binary solution:', r_custom.best_sample.astype(int))
print(f'Best QUBO energy    : {r_custom.best_energy:.6f}')

plot_trotter_energies(r_custom, Q_custom, 'Custom QUBO')

In [ ]:
# ── Brute-force check (n ≤ 20 only) ──────────────────────────────────────────
if n_custom <= 20:
    best_bf_e = np.inf
    best_bf_x = None
    for bits in range(2**n_custom):
        x_bf = np.array([(bits >> k) & 1 for k in range(n_custom)], dtype=float)
        e = qubo_energy(Q_custom, x_bf)
        if e < best_bf_e:
            best_bf_e, best_bf_x = e, x_bf
    gap = r_custom.best_energy - best_bf_e
    print(f'Brute-force optimum : {best_bf_e:.6f}  x={best_bf_x.astype(int)}')
    print(f'FPGA result         : {r_custom.best_energy:.6f}')
    print(f'Optimality gap      : {gap:.6f}  ({"OPTIMAL" if gap < 1e-4 else "suboptimal"})')
else:
    print('Too large for brute force.')

---
## Offline validation with SQASimulator

Use `SQASimulator` on your laptop to verify your Q matrix and tune
parameters **before** running on the board. It uses the exact same update
rule as the Opt5 kernel, so results should be comparable.

In [ ]:
# Drop-in replacement for SQASolver — no FPGA needed
sim = SQASimulator(num_trotters=8, seed=42)
r_sim = sim.solve(
    Q_custom,
    iters=ITERS, beta_start=BETA_START, beta_end=BETA_END, gamma_start=GAMMA_START
)

print('Simulator best energy :', r_sim.best_energy)
print('FPGA     best energy  :', r_custom.best_energy)

---
## Annealing schedule explorer

Visualise how `Jperp` (inter-trotter coupling) and `Beta` (inverse temperature)
evolve over the annealing schedule. As `Beta` rises, the system freezes;
as `Jperp` rises, trotters lock together into the classical solution.

In [ ]:
from sqa_solver import jperp

iters_plot = ITERS
beta_vals  = np.linspace(BETA_START, BETA_END, iters_plot)
gamma_vals = GAMMA_START * (1 - np.arange(iters_plot) / iters_plot)
jp_vals    = np.array([jperp(g, 8, b) for g, b in zip(gamma_vals, beta_vals)])

fig, axes = plt.subplots(1, 3, figsize=(14, 3))

axes[0].plot(beta_vals)
axes[0].set_title('Beta (inverse temperature)')
axes[0].set_xlabel('Iteration')

axes[1].plot(gamma_vals, color='orange')
axes[1].set_title('Gamma (transverse field)')
axes[1].set_xlabel('Iteration')

axes[2].plot(jp_vals, color='green')
axes[2].set_title('Jperp (inter-trotter coupling)')
axes[2].set_xlabel('Iteration')
axes[2].set_yscale('log')

plt.suptitle(f'Annealing schedule  (iters={iters_plot}, beta_end={BETA_END}, G0={GAMMA_START})',
             y=1.02)
plt.tight_layout()
plt.show()